#  ⭐ `fat_medicamentos`


In [1]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_date_column, normalize_rows, apply_mappings, fuzzy_merge##, normalize_duration, expandir_gravidade_wide
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [2]:
path = Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO'],
      dtype='object')

In [3]:
bronze.head(2)

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,CODIGO_ATC,DETENTOR_REGISTRO,CONCENTRACAO,COMPONENTE_SUSPEITO,ACAO_ADOTADA,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,FREQUENCIA_DOSE,POSOLOGIA,DURACAO,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_MAE_PAI,NUMELO_LOTE,ATUALIZADO
0,BR-ANVISA-300000004,Suspeito,Oxacilina,Oxacillin sodium,J01CF,Teuto,None,None,None,None,None,None,250 milligram (mg),6 hora,None,4 dia,20181124,None,None,infusão intravenosa,None,5833018,2025-11-17
1,BR-ANVISA-300000005,Suspeito,Paracemol,Paracetamol,N02BE,None,None,None,Retirada do medicamento,None,None,None,None,None,None,None,20181122,20181122,None,oral,None,None,2025-11-17


# 🥈 Silver

normalized data, medium quality


In [4]:
silver = bronze.copy()
silver.shape

(552887, 23)

## ✅ 1. Missing

In [5]:
silver.isna().sum()

IDENTIFICACAO_NOTIFICACAO                            0
RELACAO_MEDICAMENTO_EVENTO                        1000
NOME_MEDICAMENTO_WHODRUG                          5159
PRINCIPIOS_ATIVOS_WHODRUG                         5649
CODIGO_ATC                                        5159
DETENTOR_REGISTRO                               426201
CONCENTRACAO                                    460259
COMPONENTE_SUSPEITO                             496737
ACAO_ADOTADA                                    267347
PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO    534294
INDICACAO_MEDDRA                                325534
INDICACAO_RELATADA_NOTIFICADOR_INICIAL          315982
DOSE                                            339554
FREQUENCIA_DOSE                                   3583
POSOLOGIA                                       275637
DURACAO                                         473049
INICIO_ADMINISTRACAO                            288439
FIM_ADMINISTRACAO                               395839
FORMA_FARM

In [6]:
hist_silver = silver.copy()
for col in silver.select_dtypes(include=['object']).columns:
    silver[col] = normalize_rows(silver[col])

In [7]:
hist_silver.isna().sum()

IDENTIFICACAO_NOTIFICACAO                            0
RELACAO_MEDICAMENTO_EVENTO                        1000
NOME_MEDICAMENTO_WHODRUG                          5159
PRINCIPIOS_ATIVOS_WHODRUG                         5649
CODIGO_ATC                                        5159
DETENTOR_REGISTRO                               426201
CONCENTRACAO                                    460259
COMPONENTE_SUSPEITO                             496737
ACAO_ADOTADA                                    267347
PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO    534294
INDICACAO_MEDDRA                                325534
INDICACAO_RELATADA_NOTIFICADOR_INICIAL          315982
DOSE                                            339554
FREQUENCIA_DOSE                                   3583
POSOLOGIA                                       275637
DURACAO                                         473049
INICIO_ADMINISTRACAO                            288439
FIM_ADMINISTRACAO                               395839
FORMA_FARM

In [8]:
hist_silver.shape

(552887, 23)

## ✅ 2. datas - 'INICIO_ADMINISTRACAO', 'FIM_ADMINISTRACAO'



In [9]:
colunas_data = ["INICIO_ADMINISTRACAO", "FIM_ADMINISTRACAO"]

In [10]:
hist_silver[colunas_data].info()
hist_silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 2 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   INICIO_ADMINISTRACAO  264448 non-null  object
 1   FIM_ADMINISTRACAO     157048 non-null  object
dtypes: object(2)
memory usage: 8.4+ MB


,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO
0,20181124,None
1,20181122,20181122
2,20181103,20181115
3,20181016,20181115
4,20181024,None


In [11]:
for col in colunas_data:
    if col in hist_silver.columns:
        hist_silver[col] = normalize_date_column(hist_silver[col])

In [12]:
hist_silver[colunas_data].info()
hist_silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 2 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   INICIO_ADMINISTRACAO  220969 non-null  datetime64[ns]
 1   FIM_ADMINISTRACAO     142767 non-null  datetime64[ns]
dtypes: datetime64[ns](2)
memory usage: 8.4 MB


,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO
0,2018-11-24,NaT
1,2018-11-22,2018-11-22
2,2018-11-03,2018-11-15
3,2018-10-16,2018-11-15
4,2018-10-24,NaT


In [13]:
hist_silver.shape

(552887, 23)

## ✅ 3. Mappings

- RELACAO_MED_EVENTO
- COMPONENTE_SUSPEITO
- ACAO_ADOTADA

In [14]:
# RELACAO_MEDICAMENTO_EVENTO
relacao_medicamento_evento_map = {
    "SUSPEITO": 1,
    "CONCOMITANTE": 2,
    "MEDICAMENTO NAO ADMINISTRADO": 3,
    "INTERACAO": 4,
}
# COMPONENTE_SUSPEITO
componente_suspeito_map = {
    "PRINCIPIO ATIVO": 1, "EXCIPIENTE, NAO CLASSIFICADO": 2, "SOLVENTE": 3, "CORANTE": 4,
    "CONSERVANTE": 5, "AGENTE FLAVORIZADOR": 6, "EXCESSO PERCENTUAL": 7, "ANTIOXIDANTE": 8,
    "ESTABILIZANTE": 9
}

# ACAO_ADOTADA
acao_adotada_map = {
    "RETIRADA DO MEDICAMENTO": 1, "SEM ALTERACAO DA DOSE": 2, "NAO APLICAVEL": 3,
    "REDUCAO DA DOSE": 4, "AUMENTO DA DOSE": 5
}
mappings_config = [
    {
        "kind": "dict",
        "col": "RELACAO_MEDICAMENTO_EVENTO",
        "map": relacao_medicamento_evento_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": True,
    },
    {
        "kind": "dict",
        "col": "COMPONENTE_SUSPEITO",
        "map": componente_suspeito_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": True,
    },
    {
        "kind": "dict",
        "col": "ACAO_ADOTADA",
        "map": acao_adotada_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": True,
    }]

In [15]:
hist_silver = apply_mappings(hist_silver, mappings_config)

In [16]:
hist_silver.shape

(552887, 26)

In [17]:
hist_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 26 columns):
 #   Column                                        Non-Null Count   Dtype         
---  ------                                        --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                     552887 non-null  object        
 1   NOME_MEDICAMENTO_WHODRUG                      547728 non-null  object        
 2   PRINCIPIOS_ATIVOS_WHODRUG                     547238 non-null  object        
 3   CODIGO_ATC                                    547728 non-null  object        
 4   DETENTOR_REGISTRO                             126686 non-null  object        
 5   CONCENTRACAO                                  92628 non-null   object        
 6   PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO  18593 non-null   object        
 7   INDICACAO_MEDDRA                              227353 non-null  object        
 8   INDICACAO_RELATADA_NOTIFICADOR_INICIAL        236905 n

## ✅ 4. PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO

In [18]:
from utils import expandir_lista_wide

In [19]:
hist_silver = expandir_lista_wide(
    hist_silver,
    col="PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO",
    prefix="PROB_ADIC_",   # se quiser um prefixo mais curto/bonito
)


In [20]:

hist_silver.filter(like="PROB_ADIC_").head(2)

,PROB_ADIC_ABUSO,PROB_ADIC_ERRO_DE_MEDICACAO,PROB_ADIC_EXPOSICAO_OCUPACIONAL,PROB_ADIC_FALSIFICACAO,PROB_ADIC_LOTES_TESTADOS_E_DENTRO_DAS_ESPECIFICACOES,PROB_ADIC_LOTES_TESTADOS_E_FORA_DAS_ESPECIFICACOES,PROB_ADIC_MEDICAMENTO_TOMADO_FORA_DA_DATA_DE_VALIDADE,PROB_ADIC_MEDICAMENTO_TOMADO_PELO_PAI,PROB_ADIC_SUPERDOSE,PROB_ADIC_USO_INCORRETO,PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO
0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0


In [21]:
hist_silver.shape

(552887, 37)

In [22]:
hist_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 37 columns):
 #   Column                                                 Non-Null Count   Dtype         
---  ------                                                 --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                              552887 non-null  object        
 1   NOME_MEDICAMENTO_WHODRUG                               547728 non-null  object        
 2   PRINCIPIOS_ATIVOS_WHODRUG                              547238 non-null  object        
 3   CODIGO_ATC                                             547728 non-null  object        
 4   DETENTOR_REGISTRO                                      126686 non-null  object        
 5   CONCENTRACAO                                           92628 non-null   object        
 6   PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO           552887 non-null  object        
 7   INDICACAO_MEDDRA                                       2

## ✅ 5. DURACAO

In [23]:
from utils import normalize_duracao

In [24]:
hist_silver.shape

(552887, 37)

In [25]:
hist_silver = normalize_duracao(hist_silver, coluna="DURACAO")

In [26]:
hist_silver.head()

,IDENTIFICACAO_NOTIFICACAO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,CODIGO_ATC,DETENTOR_REGISTRO,CONCENTRACAO,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,FREQUENCIA_DOSE,POSOLOGIA,DURACAO,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_MAE_PAI,NUMELO_LOTE,ATUALIZADO,RELACAO_MEDICAMENTO_EVENTO_VALOR,RELACAO_MEDICAMENTO_EVENTO_CHAVE,COMPONENTE_SUSPEITO_VALOR,COMPONENTE_SUSPEITO_CHAVE,ACAO_ADOTADA_VALOR,ACAO_ADOTADA_CHAVE,PROB_ADIC_ABUSO,PROB_ADIC_ERRO_DE_MEDICACAO,PROB_ADIC_EXPOSICAO_OCUPACIONAL,PROB_ADIC_FALSIFICACAO,PROB_ADIC_LOTES_TESTADOS_E_DENTRO_DAS_ESPECIFICACOES,PROB_ADIC_LOTES_TESTADOS_E_FORA_DAS_ESPECIFICACOES,PROB_ADIC_MEDICAMENTO_TOMADO_FORA_DA_DATA_DE_VALIDADE,PROB_ADIC_MEDICAMENTO_TOMADO_PELO_PAI,PROB_ADIC_SUPERDOSE,PROB_ADIC_USO_INCORRETO,PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_VALOR
0,BR-ANVISA-300000004,Oxacilina,Oxacillin sodium,J01CF,Teuto,None,[],None,None,250 milligram (mg),6 hora,None,4 dia,2018-11-24,NaT,None,infusão intravenosa,None,5833018,2025-11-17,Suspeito,0,DESCONHECIDO,0,DESCONHECIDO,0,0,0,0,0,0,0,0,0,0,0,0,dia,4,4.0
1,BR-ANVISA-300000005,Paracemol,Paracetamol,N02BE,None,None,[],None,None,None,None,None,None,2018-11-22,2018-11-22,None,oral,None,None,2025-11-17,Suspeito,0,DESCONHECIDO,0,Retirada do medicamento,0,0,0,0,0,0,0,0,0,0,0,0,desconhecido,0,NaN
2,BR-ANVISA-300000007,Diamox,Acetazolamide sodium,C03|N03AX|S01EC,None,None,[],None,None,250 milligram (mg),6 hora,250mg a cada 6 horas,None,2018-11-03,2018-11-15,None,oral,None,None,2025-11-17,Suspeito,0,DESCONHECIDO,0,Retirada do medicamento,0,0,0,0,0,0,0,0,0,0,0,0,desconhecido,0,NaN
3,BR-ANVISA-300000007,Hidantal,Phenytoin,N03AB,None,None,[],None,None,None,None,100mg a cada 8 horas,None,2018-10-16,2018-11-15,None,oral,None,None,2025-11-17,Suspeito,0,DESCONHECIDO,0,Retirada do medicamento,0,0,0,0,0,0,0,0,0,0,0,0,desconhecido,0,NaN
4,BR-ANVISA-300000008,Nitroglicerina,Glyceryl trinitrate,C01DA|C05AE,Cristália,None,[],None,None,None,None,None,None,2018-10-24,NaT,None,infusão intravenosa,None,None,2025-11-17,Suspeito,0,DESCONHECIDO,0,DESCONHECIDO,0,0,0,0,0,0,0,0,0,0,0,0,desconhecido,0,NaN


In [27]:
hist_silver.shape

(552887, 40)

In [28]:
hist_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 40 columns):
 #   Column                                                 Non-Null Count   Dtype         
---  ------                                                 --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                              552887 non-null  object        
 1   NOME_MEDICAMENTO_WHODRUG                               547728 non-null  object        
 2   PRINCIPIOS_ATIVOS_WHODRUG                              547238 non-null  object        
 3   CODIGO_ATC                                             547728 non-null  object        
 4   DETENTOR_REGISTRO                                      126686 non-null  object        
 5   CONCENTRACAO                                           92628 non-null   object        
 6   PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO           552887 non-null  object        
 7   INDICACAO_MEDDRA                                       2

In [29]:
#hist_silver = hist_silver.drop(columns=["DURACAO"])

## ✅ 6. FORMA_FARMACEUTICA

In [30]:
hist_dim_forma_farmaceutica = pd.read_parquet(Path(project_root) / "data/02_silver/hist_dim_forma_farmaceutica/hist_dim_forma_farmaceutica.parquet")
hist_dim_forma_farmaceutica[['FORMA_FARMACEUTICA','FORMA_FARMACEUTICA_CHAVE', 'FORMA_FARMACEUTICA_VALOR']].drop_duplicates().head()

,FORMA_FARMACEUTICA,FORMA_FARMACEUTICA_CHAVE,FORMA_FARMACEUTICA_VALOR
0,Adesivo Transdérmico,1,Adesivo transdermico
1,adesivo,1,Adesivo transdermico
2,adesivo transdérmico,1,Adesivo transdermico
3,Adesivo,1,Adesivo transdermico
5,ADESIVO,1,Adesivo transdermico


In [31]:
silver.shape

(552887, 23)

In [32]:
hist_silver.shape

(552887, 40)

In [33]:
hist_silver = fuzzy_merge(
    hist_dim_forma_farmaceutica,
    hist_silver,
    on="FORMA_FARMACEUTICA",
    threshold=95,
    suffix="",
    dedupe_on=False,
)
hist_silver['FORMA_FARMACEUTICA_VALOR'] = hist_silver['FORMA_FARMACEUTICA_VALOR'].fillna('DESCONHECIDO')
hist_silver['FORMA_FARMACEUTICA_CHAVE'] = hist_silver['FORMA_FARMACEUTICA_CHAVE'].fillna(0) 

In [34]:
hist_silver.shape

(552887, 44)

In [35]:
hist_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 44 columns):
 #   Column                                                 Non-Null Count   Dtype         
---  ------                                                 --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                              552887 non-null  object        
 1   NOME_MEDICAMENTO_WHODRUG                               547728 non-null  object        
 2   PRINCIPIOS_ATIVOS_WHODRUG                              547238 non-null  object        
 3   CODIGO_ATC                                             547728 non-null  object        
 4   DETENTOR_REGISTRO                                      126686 non-null  object        
 5   CONCENTRACAO                                           92628 non-null   object        
 6   PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO           0 non-null       float64       
 7   INDICACAO_MEDDRA                                       2

In [36]:
#hist_silver = hist_silver.drop(columns=["FORMA_FARMACEUTICA"])

## ✅ 7. DETENTOR_REGISTRO

In [37]:
hist_dim_detentor_registro = pd.read_parquet(Path(project_root) / "data/02_silver/hist_dim_detentor_registro/hist_dim_detentor_registro.parquet")
hist_dim_detentor_registro[['DETENTOR_REGISTRO','DETENTOR_REGISTRO_CHAVE', 'DETENTOR_REGISTRO_VALOR']].drop_duplicates().head()

,DETENTOR_REGISTRO,DETENTOR_REGISTRO_CHAVE,DETENTOR_REGISTRO_VALOR
0,ABACUS MEDICINE,1,ABACUS MEDICINE
1,ABBOLT,2,ABBOT
2,ABBOT,2,ABBOT
3,ABBOT BRASIL,2,ABBOT
4,ABBOT L: 0071/20 V: 31/01/2022,2,ABBOT


In [38]:
silver.shape

(552887, 23)

In [39]:
hist_silver = fuzzy_merge(
    hist_dim_detentor_registro,
    hist_silver,
    on="DETENTOR_REGISTRO",
    threshold=95,
    suffix="",
    dedupe_on=False,
)
hist_silver['DETENTOR_REGISTRO_VALOR'] = hist_silver['DETENTOR_REGISTRO_VALOR'].fillna('DESCONHECIDO')
hist_silver['DETENTOR_REGISTRO_CHAVE'] = hist_silver['DETENTOR_REGISTRO_CHAVE'].fillna(0) 

In [40]:
hist_silver.shape

(552887, 49)

In [41]:
silver.shape

(552887, 23)

In [42]:
#hist_silver = hist_silver.drop(columns=["DETENTOR_REGISTRO"])

## ✅ 8. VIA_ADM

In [43]:
from utils.med_normalize_via_adm import normalizar_via_fuzzy


In [44]:
hist_silver["VIA_ADMINISTRACAO_CHAVE_VALOR"] = (
    hist_silver["VIA_ADMINISTRACAO"]
    .astype(str)
    .apply(lambda txt: normalizar_via_fuzzy(
        txt,
        score_threshold=80,
        return_numeric=True,
    ))
)

hist_silver["VIA_ADMINISTRACAO_MAE_PAI_VALOR"] = (
    hist_silver["VIA_ADMINISTRACAO_MAE_PAI"]
    .astype(str)
    .apply(lambda txt: normalizar_via_fuzzy(
        txt,
        score_threshold=80,
        return_numeric=True,
    ))
) 


In [45]:

hist_silver.shape


(552887, 51)

In [46]:
hist_silver.shape

(552887, 51)

## ✅ 9. CONCENTRACAO

In [47]:
from utils.med_normalize_concentracao import normalize_concentracao


In [48]:
hist_silver = normalize_concentracao(hist_silver, col="CONCENTRACAO")

In [49]:
hist_silver.shape

(552887, 56)

## ✅ 10. DOSE

In [50]:
from utils.med_normalize_dose import normalize_dose

In [51]:
hist_silver = normalize_dose(hist_silver, col="DOSE")


In [52]:
hist_silver.shape

(552887, 59)

## 13. INDICACAO_MEDDRA

In [53]:
hist_silver.INDICACAO_MEDDRA.value_counts().head()

INDICACAO_MEDDRA
USO DE MEDICAMENTO PARA INDICAÇÃO DESCONHECIDA    19595
PRODUTO USADO PARA INDICAÇÃO DESCONHECIDA         16515
DOENÇA DE CROHN                                    6875
ARTRITE REUMATOIDE                                 6207
HIPERTENSÃO                                        4399
Name: count, dtype: int64

## INDICACAO_RELATADA_NOTIFICADOR_INICIAL

In [54]:
hist_silver.INDICACAO_RELATADA_NOTIFICADOR_INICIAL.value_counts().head()

INDICACAO_RELATADA_NOTIFICADOR_INICIAL
PRODUCT USED FOR UNKNOWN INDICATION    11420
DRUG USE FOR UNKNOWN INDICATION         6094
DOENÇA DE CROHN                         3180
DIABETES                                3010
HIPERTENSÃO                             2838
Name: count, dtype: int64

## ✅ 12. Desagregar atc_code_level_4

In [55]:
from utils import desagrupar_colunas_pipe

In [56]:
colunas_para_desagrupar = ['ATC_CODE_LEVEL_4']
print(f"Shape antes da desagregação: {hist_silver.shape}") 
hist_silver_dedup = desagrupar_colunas_pipe(hist_silver.rename(columns={'CODIGO_ATC': 'ATC_CODE_LEVEL_4'}), colunas_para_desagrupar)
print(f"Shape depois da desagregação: {hist_silver_dedup.shape}") 

Shape antes da desagregação: (552887, 59)
Shape depois da desagregação: (748297, 59)


# fim

In [57]:
hist_silver_dedup.to_parquet(Path(project_root) / "data/03_gold/fat_medicamentos/fat_medicamentos.parquet", index=False)